In the following, we report a template for the final project. Feel free to modify it to better fit your own project.

# **Project Title**
**Per-Head Heavy-Hitter Oracle (PH-H2O)**

## **Abstract**
KV cache compression has been widely applied across domains such as natural language processing, computer vision, and other tasks. In this project, I reproduced four baselines — full cache (no eviction), LRU KV cache, Heavy-Hitter Oracle (H2O per-layer), and random eviction cache — and compared them against an H2O variant using a per-head KV cache method. Evaluation was conducted on an NVIDIA H100 GPU with 80 GB of memory on the CCDB Trillium cluster. The original H2O paper claims that retaining only 20% of the KV cache achieves approximately 80% accuracy or coverage on the CNN/DailyMail dataset. Our evaluation reproduces this result. Furthermore, the proposed per-head KV cache method achieves a 6% improvement over the per-layer H2O baseline at a 20% budget.



## **1. Introduction**

### **1.1 Problem Statement and Motivations**

Existing KV cache eviction policies, such as the standard H2O (Heavy-Hitter Oracle), typically compute eviction scores by aggregating importance metrics across all attention heads within a layer. While computationally efficient, this per-layer agreement creates a "**consensus bottleneck.**"

Because different heads attend to different semantic or syntactic features, a specific token (e.g., index 9) may be a "Heavy-Hitter" for Head 1 but irrelevant for Heads 2 and 3. In a shared eviction scheme, the low scores from Heads 2 and 3 can lead to the eviction of index 9, causing a localized loss of critical information for Head 1 and ultimately degrading model accuracy.

<figure style="text-align: center">
  <img src="../resource/attention.png" width="900">
  <figcaption>Figure 1: Attention mechanism</figcaption>
</figure>

Here is an example in the figure 1. The sequence "The cat sat on the mat near the TV". Assuming 4 heads are focusing on suj. and verb. , head 2 on article and det., head 3 on location and head 4 focusing on preposition. The per layer score will keep "The" token in Top 1 position but it is not top1 for all heads just an average result. Therefore, by using per head cache, each head can just reserve the corresponding top k value instead of using per-layer consensus.

### **1.2 Proposed solution**
To address this, I propose an incremental per-head eviction strategy ---- **Per-Head Heavy-Hitter Oracle (PH-H2O)**. Instead of a shared eviction index, each attention head maintains its own independent KV cache metadata and eviction policy.

This method has following advantages :
By decoupling the eviction logic to ensure that "specialist" heads can retain

1.   **Finer granularity**.the tokens most relevant to their specific functional role without being overridden by the "average" importance of the layer.
2.   **Same budget cache size but accuracy is improved**. Even when the total cache capacity (memory footprint) remains identical to the baseline, the information density of the cache increases because each head's budget is spent only on its most relevant tokens.
3.   **Minimal overhead.** Since H2O scores are already calculated per-head during the attention pass, the primary cost is maintaining head-specific bitmasks or index lists, which is negligible compared to the accuracy recovery.



<div style="display:flex; justify-content:center;">
<table style="border-collapse:collapse; font-family:'Segoe UI',Arial,sans-serif; font-size:14px;">
  <thead>
    <tr style="background:#1a73e8; color:#fff;">
      <th style="padding:10px 14px; text-align:left; font-weight:600; white-space:nowrap;">Policy</th>
      <th style="padding:10px 14px; text-align:left; font-weight:600; white-space:nowrap;">Granularity</th>
      <th style="padding:10px 14px; text-align:left; font-weight:600; white-space:nowrap;">Eviction Logic</th>
      <th style="padding:10px 14px; text-align:left; font-weight:600; white-space:nowrap;">Memory Footprint</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background:#fff;">
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;"><code>Full Cache</code></td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; text-align:center;">—</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">No eviction</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Full sequence</td>
    </tr>
    <tr style="background:#f8f9fa;">
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;"><code>Random</code></td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; text-align:center;">Layer</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Random selection</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Fixed budget</td>
    </tr>
    <tr style="background:#fff;">
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;"><code>Local (LRU)</code></td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; text-align:center;">Layer</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Keep recent tokens</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Fixed budget</td>
    </tr>
    <tr style="background:#f8f9fa;">
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;"><code>H2O (Standard)</code></td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; text-align:center;">Layer</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Recent + Heavy Hitter</td>
      <td style="padding:9px 14px; border-bottom:1px solid #e8eaed; white-space:nowrap;">Fixed budget</td>
    </tr>
    <tr style="background:#fff;">
      <td style="padding:9px 14px; white-space:nowrap;"><code>H2O (Per-head)</code></td>
      <td style="padding:9px 14px; text-align:center;">Head</td>
      <td style="padding:9px 14px; white-space:nowrap;">Recent + Heavy Hitter</td>
      <td style="padding:9px 14px; white-space:nowrap;">Fixed budget</td>
    </tr>
  </tbody>
</table>
</div>


**Same total budget — different selection granularity**
1. Per-layer: one global ranking loses head-specific patterns
2. Per-head: each head keeps what it actually attends to




## **2. Methodology**

<p>This project investigates KV cache compression for large language models, evaluating the Heavy Hitter Oracle (H2O) eviction strategy on <b>Llama-3.2-1B-Instruct</b> for abstractive summarization.</p>

**2.1 Model and dataset**
<p>The algorithm is evaluated on the <b>CNN/DailyMail 3.0.0 benchmark</b> using Llama-3.2-1B-Instruct — a 16-layer decoder-only model with Grouped Query Attention (GQA): 32 query heads and 8 KV heads. All experiments use a maximum input length of 4096 tokens and a fixed KV cache budget of 512 tokens.</p>

**2.2 KV cache eviction policies**
<p>four eviction strategies implemented and compared :</p>
<ul>
  <li><b>Full cache:</b> retains all KV pairs with no eviction. Upper-bound baseline.</li>
  <li><b>Random:</b> randomly selects tokens within a fixed budget. Lower-bound baseline.</li>
  <li><b>Local / sliding window:</b> retains only the most recent tokens (<code>budget = recent</code>). Pure recency-based cache with no heavy hitter tracking.</li>
  <li><b>H2O:</b> combines a recent window with attention-guided heavy hitters. Budget satisfies <code>total = HH + recent</code>.</li>
</ul>

**2.3 H2O eviction: per-layer vs per-head**
<p>By introducing a key design variant — whether heavy hitter selection operates at the layer level or the head level.</p>
<ul>
  <li><b>Per-layer:</b> attention scores are averaged across all KV heads, and a single shared top-k selection is applied uniformly to all heads. It's imple and efficient, but loses head-specific attention patterns.</li>
  <li><b>Per-head:</b> each KV head independently computes its own top-k heavy hitters, preserving head-specific linguistic patterns. The total budget is divided equally among heads: <code>per_head_budget = total_budget / num_kv_heads</code>.</li>
</ul>

<div style="display: flex; gap: 20px; justify-content: center; margin: 1rem 0;">
  <figure style="text-align: center; flex: 1;">
    <img src="../resource/Per%20layer.png" width="700">
    <figcaption>Figure 2: Per-layer KV cache demonstration</figcaption>
  </figure>
  <figure style="text-align: center; flex: 1;">
    <img src="../resource/per%20head.png" width="700">
    <figcaption>Figure 3: Per-head KV cache demonstration</figcaption>
  </figure>
</div>

<h3>GQA handling</h3>
<p>Because Llama-3.2-1B uses GQA with a 4:1 query-to-KV ratio, attention weights are reshaped and averaged over the 4 query heads sharing each KV head before computing accumulated scores:</p>
<pre><code>score = raw_score.view(NUM_KV_HEADS, GQA_GROUP, -1).mean(dim=1)</code></pre>

<h3>Prefill eviction</h3>
<p>Eviction is applied via a forward hook on <code>model.model</code>, triggering after each full forward pass. The hook inspects the <code>DynamicCache</code> and prunes KV entries whenever the sequence length exceeds the budget, keeping the top heavy hitters plus the most recent tokens.</p>





**2.4 Implementation**

Here is the flow chart of the implementation:

<div style="display: flex; gap: 24px; justify-content: center; align-items: flex-start;">
  <figure style="text-align: center; flex: 1; margin: 0;">
    <img src="../resource/flow chart.png" width="700">
    <figcaption>Figure 3: Per-head KV cache demonstration</figcaption>
  </figure>

  <figure style="text-align: center; flex: 1; margin: 0;">
    <img src="../resource/llama model.png" width="700">
    <figcaption>Figure 4: Llama 3.2-1B-Instruct and Monkey patch</figcaption>
  </figure>
</div>


In this section, you can add **text** and **figures**. For instance, it is strongly suggested to add a picture of the best machine learning model that you implemented to solve your problem (and describe it).


## **3. Experimental Setup**
Describe the datasets used for your experiments. List the machine learning techniques used to solve your problem and report the corresponding hyperparameters.

**3.1 Hardware and environment**
<p>All experiments were conducted on a single <b>NVIDIA H100 GPU with 80 GB of memory</b> on CCDB trillium cluster. The model was loaded in <code>bfloat16</code> precision using HuggingFace Transformers with <code>attn_implementation="eager"</code> to enable attention weight extraction.</p>

**3.2 Model**
<p>I use <b>Llama-3.2-1B-Instruct</b> (<code>meta-llama/Llama-3.2-1B-Instruct</code>) with the following configuration:</p>
<ul>
  <li>Hidden size: 2048, 16 transformer layers</li>
  <li>32 query heads, 8 KV heads (GQA with group size 4)</li>
</ul>

**3.3 Dataset**
<p>I evaluate on the following dataset</p>
<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 100%;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th>Dataset</th>
      <th>Source</th>
      <th>Task</th>
      <th>Input Field</th>
      <th>Reference Field</th>
      <th>max_seq_len</th>
      <th>cache_size</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>CNN/DailyMail</b> 3.0.0</td>
      <td>HuggingFace <code>cnn_dailymail</code></td>
      <td>Summarization</td>
      <td><code>article</code></td>
      <td><code>highlights</code></td>
      <td>3000</td>
      <td>512</td>
    </tr>
    <tr>
      <td><b>GovReport</b></td>
      <td>LongBench <code>gov_report</code></td>
      <td>Summarization</td>
      <td><code>context</code></td>
      <td><code>answers</code></td>
      <td>3000</td>
      <td>512</td>
    </tr>
    <tr>
      <td><b>QMSum</b></td>
      <td>LongBench <code>qmsum</code></td>
      <td>Summarization</td>
      <td><code>context</code></td>
      <td><code>answers</code></td>
      <td>3000</td>
      <td>512</td>
    </tr>
    <tr>
      <td><b>VCSum</b></td>
      <td>LongBench <code>vcsum</code></td>
      <td>Summarization</td>
      <td><code>context</code></td>
      <td><code>answers</code></td>
      <td>3000</td>
      <td>512</td>
    </tr>
  </tbody>
</table>
<p>Here is an example of <b>CNN/DailyMail 3.0.0</b> validation split. Each article is truncated to a maximum of 4096 tokens:</p>

Here is what does each record in the CNN dataset:


<code>
Article: "LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20......".


Highlights: "Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday . Young actor.......".
</code>

The prompt used is :
<code>Prompt:"Summarize this article in 2-3 sentences:".</code>

<p>A maximum of 100 new tokens can be generated.</p>

**3.4 KV cache budget**
<p>The total KV cache size is fixed at <b>512 tokens</b>. Sweep the following hyperparameters:</p>
<ul>
  <li><code>budget_ratio</code>: 0.1 to 0.8 (fraction of 512 tokens to keep)</li>
  <li><code>recent_ratio</code>: 0.1 to 0.5 (fraction reserved for the recency window)</li>
  <li>H2O strategy: <code>per_head</code> vs <code>layer_shared</code></li>
</ul>

**3.5 Baselines**
<ul>
  <li><b>Full cache:</b> no eviction, retains all tokens</li>
  <li><b>Random eviction:</b> uniformly random token selection at each budget level</li>
  <li><b>Local (sliding window):</b> retains only the most recent tokens (<code>budget_ratio = recent_ratio</code>)</li>
  <li><b>H2O (per layer): </b>shown in fig 2</li>
  <li><b>PH H2O (per head H2O):</b>shown in fig 3</li>
</ul>

**3.6 Evaluation metric**
<p>All strategies are evaluated using <b>ROUGE-L F1</b> against the reference summaries (<code>highlights</code> field) from CNN/DailyMail, computed via the <code>rouge l1 score</code> from rouge_score library.</p>



## **4. Experimental Results**

In the experiments, I choosed following dataset CNN/DailyMail and gov_report,qmsum, vcsum from LongBench.



<div style="display: flex; gap: 5px; justify-content: center; margin: 1rem 0;">
  <figure style="text-align: center; flex: 1;">
    <img src="../plots/h2o_coverage_vs_budget.png" width="900">
    <figcaption>Figure 4: budget sweep and Rouge-L score</figcaption>
  </figure>
  <figure style="text-align: center; flex: 1;">
    <img src="../plots/H2O paper CNN plots.png" width="750">
    <figcaption>Figure 5: Plots in the H2O paper</figcaption>
  </figure>
</div>

The original H2O paper reports roughly 80% of full-cache accuracy at a 20% compression ratio on the CNN/DailyMail dataset, and our reproduction achieves comparable results. 

Figure 5 presents two baselines from the origional paper in Heavy Hitter Oracle : full cache (no compression) and recent-only cache. 

Figure 4 compares five configurations: 

        **(1) random KV cache eviction**

        **(2) local/recent-only cache**

        **(3) full cache with no eviction**

        **(4) H2O with per-layer eviction (solid line)**

        **(5) H2O with per-head eviction (dashed line)**

At 80% budget (512 × 80%), the local method performs worst and random eviction second worst. As the budget decreases, H2O configurations that allocate a smaller fraction to the recent window and a larger fraction to heavy hitters achieve higher ROUGE-L scores at the same total budget, illustrating the importance of the heavy-hitter selection mechanism. Across all budget levels, per-head eviction consistently outperforms per-layer eviction by 4–6 ROUGE-L points. Although all compressed methods fall below the full-cache baseline, the results clearly demonstrate the trade-off between total budget size and the allocation split between recent tokens and heavy hitters.




If we take the best among all different total budget we got :
  <figure style="text-align: center; flex: 1;">
    <img src="../plots/coverage_vs_budget.png" width="750">
    <figcaption>Figure 6: Roug-L vesus Budget for all methods</figcaption>
  </figure>


We can see that, we can further use the ratio between recent and heavy hitter portion ratio to fine tune and get the best performance of all different total budget, but after 20% compression ratio the accuracy drops sharply.


  <figure style="text-align: center; flex: 1;">
    <img src="../plots/rouge_all_configs_bar.png" width="750">
    <figcaption>Figure 6: All Roug-L for all methods</figcaption>
  </figure>

**4.1 Latency test**

I did a latency test, but the timer starts before the eviction. So this latency is prefill + eviction latency.
So, because of my mistake the prefill is the dominant part in the latency, so this plots shows all methods have similar latency which is incorrect, but I don't have time to rerun all experiments to get new latency plots.

  <figure style="text-align: center; flex: 1;">
    <img src="../plots/latency_memory_vs_budget.png" width="1200">
    <figcaption>Figure 7: Latency comparision</figcaption>
  </figure>

**4.2 Qualitative Results: KV Cache Eviction on CNN/DailyMail**

Then I am very curious about the output sentence, so I take a test sample which is the index 0 for validation set Validation[0] and checked the outputs:

**Here is the test sample:**

**Article**

(CNN) Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give one of her kidneys to a stranger, her generosity paired up with big data — resulting in six patients receiving transplants.
"I thought I was going to help this one person who I don't know, but the fact that so many people can have a life extension, that's pretty big," Broussard told CNN affiliate KGO.
The power that multiplied Broussard's gift was data processing of genetic profiles from donor-recipient pairs. Working on a swapping principle scaled to a much higher level, California Pacific Medical Center in San Francisco coordinated five surgeons, dozens of support staff, and more than 40 personnel to perform surgeries on 12 people — extracting six kidneys from donors and implanting them into six recipients.
A computer programmer named David Jacobs, himself a kidney transplant recipient, created MatchGrid, a program that genetically matches donor pairs or chains in weeks rather than months. With altruistic donor Broussard entering the chain, the number of viable matching options jumped from 3–4 to 140.

**Reference Summary**

Zully Broussard decided to give a kidney to a stranger.A new computer program helped her donation spur transplants for six kidney patients.

---

| Config | Strategy | ROUGE-L |
|--------|----------|---------|
| `full` | per_head | 0.1714 |
| `h2o_b7_r5` | layer_shared | 0.1818 |
| `h2o_b6_r3` | layer_shared | 0.1667 |
| `h2o_b5_r1` | layer_shared | 0.1584 |
| `random_b1` | per_head | 0.0488 |
| `local_b4` | layer_shared | 0.0325 |
| `local_b1` | layer_shared | 0.0165 |

---

### Full Cache — `full` · per_head · ROUGE-L = 0.1714

```
A woman named Zully Broussard selflessly donated one of her kidneys to a stranger,
which sparked a chain reaction of six transplants, with her kidney being matched with
a recipient who was paired with a donor. This rare and complex process, known as paired
donation, uses genetic profiles to match compatible donors and recipients, and has been
made possible by the use of computer programs like MatchGrid...
```

---

### H2O — `h2o_b7_r5` · layer_shared · ROUGE-L = 0.1818

```
A selfless act of generosity led to a chain of six kidney transplants, with the donor's
kidney being matched with a stranger's recipient. The chain was made possible by a
computer program called MatchGrid, which uses genetic data to find compatible donors
and recipients. The program was developed by a programmer who used his skills to create
a system that can quickly match up donors and recipients...
```

---

### H2O — `h2o_b6_r3` · layer_shared · ROUGE-L = 0.1667

```
A woman named Zully Broussard selflessly donated one of her kidneys to a stranger,
which was then matched with another donor to save the life of a patient. The process,
known as paired donation, uses genetic profiles to find compatible donors and recipients,
and has been successful in pairing donors and recipients in a remarkable chain of surgeries...
```

---

### H2O — `h2o_b5_r1` · layer_shared · ROUGE-L = 0.1584

```
A selfless act of generosity led to a remarkable chain of events, where a kidney was
successfully transplanted from one person to another through a process called paired
donation. The process involves finding a compatible donor and recipient, and then using
data processing to match them...
```

---

### Random Eviction — `random_b1` · per_head · ROUGE-L = 0.0488

```
A A A A A A A A A A A A A AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA
AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA AA ...
```

Severe repetition — random eviction at 10% budget destroys output coherence entirely.

---

### Local/Sliding Window — `local_b4` · layer_shared · ROUGE-L = 0.0325

```
A nice Shirley Williams her my my my my my my my my my my my my my my
my my my my my my my my my my my my my my my my my my my my my my ...
```

Degenerate output — sliding window discards long-range context, causing the model
to latch onto the last few tokens in the article.

---

### Local/Sliding Window — `local_b1` · layer_shared · ROUGE-L = 0.0165

```
A angel my divine divine divine divine divine divine divine divine divine
divine divine divine divine divine divine divine divine divine divine ...
```

Complete failure — smallest budget combined with recent-only eviction produces
fully incoherent output.

---

### Key Observations

- H2O at high budget (`b7_r5`) slightly exceeds full cache (0.1818 vs. 0.1714),
  suggesting heavy-hitter selection can filter noise from the full context.
- Heavy hitter allocation is critical: `h2o_b5_r1` (50% budget, small recent window)
  outperforms `local_b4` (40% budget, all recent) by 4.8x in ROUGE-L.
- Random eviction collapses at low budget, producing no coherent output.
- Local/sliding window is the most fragile method, failing even at moderate budgets
  due to loss of long-range narrative context.



  <figure style="text-align: center; flex: 1;">
    <img src="../plots/prediction_comparison_sample0.png" width="100%">
    <figcaption>Figure 7: Complete rouge-L score for sample 0 in validation set</figcaption>
  </figure>




**5. Code Section**



Import all dependencies;

Treansformers version is **5.0.0**

Torch version is **2.10.0**

**Model**:*Llama-3.2-1B-Instruct*

**Dataset**:*cnn_dailymail-3.0.0 or longbench*



Model set up and arguments parse
Command format is :

The script requires four key inputs to run:


- **--dataset**: Which dataset to use.(eg."cnn_dailymail", "gov_report", "qmsum", "vcsum")

- **--num-samples**: How many data points to test.

- **--split**: Which part of the dataset to use (e.g., **validation**).

- **--kv-mode**: The specific compression configuration (e.g., **h2o_b3_r1** this means 30% of the cache size, 10% of cache size used as recent portion).

- **--h2o-strategy**: Whether to calculate importance per individual head (**per_head**) or shared across the layer (**layer_shared**).

- **--max-seq-len**: max input sequence length.

- **--cache-size**: max cache budget( <b>100% budget</b> ).


Here is the playable code section:

**Please create a .env file which contains your token for huggingface**

HF_TOKEN=your token

HF_HUB_OFFLINE=0

TRANSFORMERS_OFFLINE=0

HF_DATASETS_OFFLINE=0


Please download all the dependency with **"pip install torch numpy matplotlib python-dotenv datasets transformers tqdm"**

In [ ]:
# Install/import required libraries
import torch
import sys
from pathlib import Path
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import os
import inspect
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# Load environment variables from .env file, please add your token in the .env file as OPENAI_API_KEY=your_token_here
notebook_dir = Path.cwd()
env_path = notebook_dir / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print(f"✓ Loaded .env from {env_path}")
else:
    print(f".env not found at {env_path}")
    
def _patch_torch_autocast_compat():
    try:
        params = inspect.signature(torch.is_autocast_enabled).parameters
    except (TypeError, ValueError):
        params = {}
    if "device_type" not in params:
        _orig = torch.is_autocast_enabled
        def _compat(device_type=None):
            return _orig()
        torch.is_autocast_enabled = _compat
        print("[Compat] Patched torch.is_autocast_enabled for transformers compatibility")

_patch_torch_autocast_compat()

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

✓ Loaded .env from /scratch/gjp1993/Conversational_AI_Project/CONVERSATIONAL_AI_project/notebooks/.env
PyTorch: 2.3.1
CUDA Available: True
GPU: NVIDIA H100 80GB HBM3
GPU Memory: 79.2 GB


If everything goes will you should be able to see following print information:

✓ Loaded .env from /scratch/gjp1993/Conversational_AI_Project/CONVERSATIONAL_AI_project/notebooks/.env
PyTorch: 2.3.1
CUDA Available: True
GPU: NVIDIA H100 80GB HBM3
GPU Memory: 79.2 GB

**Configuration for program, because this suppossed to take inline arguments from CLI but here is a demo so we just hardcode all configurations here.**

In [ ]:
# ==================== CONFIGURATION ====================

DATASET = "cnn_dailymail"  #   "cnn_dailymail", "gov_report", "qmsum", "vcsum"
SPLIT = "validation"
SAMPLE_IDX = 0  # Which sample to test (0 = first sample), -1 means all samples

# Model configuration
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LEN = 3000
CACHE_SIZE = 512
MAX_NEW_TOKENS = 100

# HuggingFace token from the environment variable
HF_TOKEN = os.getenv("HF_TOKEN", "")

# ---- Define multiple configs to compare ----
CONFIGS = [
    {"kv_mode": "full",      "h2o_strategy": "per_head"},
    {"kv_mode": "random_b6", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b5_r1", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b7_r2", "h2o_strategy": "per_layer"},
    {"kv_mode": "h2o_b7_r2", "h2o_strategy": "per_head"},
]

print(f"Dataset:  {DATASET}  (split={SPLIT}, sample={SAMPLE_IDX})")
print(f"Seq len:  {MAX_SEQ_LEN}  |  Cache size: {CACHE_SIZE}  |  Max new tokens: {MAX_NEW_TOKENS}")
print(f"HF Token: {'✓ Loaded' if HF_TOKEN else '✗ Not found'}")
print(f"\nConfigs to run ({len(CONFIGS)}):")
for i, c in enumerate(CONFIGS):
    print(f"  [{i+1}] kv_mode={c['kv_mode']:20s}  strategy={c['h2o_strategy']}")

Dataset:  cnn_dailymail  (split=validation, sample=0)
Seq len:  3000  |  Cache size: 512  |  Max new tokens: 100
HF Token: ✓ Loaded

Configs to run (5):
  [1] kv_mode=full                  strategy=per_head
  [2] kv_mode=random_b6             strategy=per_head
  [3] kv_mode=h2o_b5_r1             strategy=per_head
  [4] kv_mode=h2o_b7_r2             strategy=per_layer
  [5] kv_mode=h2o_b7_r2             strategy=per_head


If everything goes will you should be able to see following print information:

Dataset:  cnn_dailymail  (split=validation, sample=0)
Seq len:  3000  |  Cache size: 512  |  Max new tokens: 100
HF Token: ✓ Loaded

Configs to run (5):
  [1] kv_mode=full                  strategy=per_head
  [2] kv_mode=random_b6             strategy=per_head
  [3] kv_mode=h2o_b5_r1             strategy=per_head
  [4] kv_mode=h2o_b7_r2             strategy=per_layer
  [5] kv_mode=h2o_b7_r2             strategy=per_head

**Load Dataset**

In [4]:
# Dataset configurations (aligned with h20.py)
def load_dataset_config(dataset_name):
    """Load dataset configuration. Aligned with h20.py."""
    configs = {
        "cnn_dailymail": {
            "loader": lambda: load_dataset("cnn_dailymail", "3.0.0"),
            "text_field": "article",
            "summary_field": "highlights",
            "prompt_template": "Summarize this article in 2-3 sentences:\n\nArticle: {text}\n\nSummary:",
            "max_seq_len": 3000,
            "cache_size": 512,
        },
        "gov_report": {
            "loader": lambda: load_dataset("THUDM/LongBench", "gov_report"),
            "text_field": "context",
            "summary_field": "answers",
            "prompt_template": "Summarize the government report in 2-3 sentences:\n\nDocument: {text}\n\nSummary:",
            "max_seq_len": 3000,
            "cache_size": 512,
        },
        "qmsum": {
            "loader": lambda: load_dataset("THUDM/LongBench", "qmsum"),
            "text_field": "context",
            "summary_field": "answers",
            "prompt_template": "Summarize the meeting in 2-3 sentences:\n\nTranscript: {text}\n\nSummary:",
            "max_seq_len": 3000,
            "cache_size": 512,
        },
        "vcsum": {
            "loader": lambda: load_dataset("THUDM/LongBench", "vcsum"),
            "text_field": "context",
            "summary_field": "answers",
            "prompt_template": "Summarize this video content in 2-3 sentences:\n\nTranscript: {text}\n\nSummary:",
            "max_seq_len": 3000,
            "cache_size": 512,
        },
    }
    
    if dataset_name not in configs:
        print(f"ERROR: Unknown dataset '{dataset_name}'. Choose: {list(configs.keys())}")
        return None
    
    return configs[dataset_name]

print(f"Loading {DATASET}...")
cfg = load_dataset_config(DATASET)
if cfg is None:
    print("Dataset not found!")
else:
    ds = cfg["loader"]()
    print(f"Splits: {list(ds.keys())}")
    print(f"Samples in {SPLIT}: {len(ds[SPLIT])}")

    # Load sample
    sample = ds[SPLIT][SAMPLE_IDX]
    text = sample[cfg["text_field"]]
    reference = sample[cfg["summary_field"]]
    reference_str = reference[0] if isinstance(reference, list) else str(reference)

    print(f"\nText length: {len(text)} characters")
    print(f"Reference: {reference_str[:100]}...")

Loading cnn_dailymail...
Splits: ['train', 'validation', 'test']
Samples in validation: 13368

Text length: 4290 characters
Reference: Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation ...


If everything goes will you should be able to see following print information:

Loading cnn_dailymail...
Splits: ['train', 'validation', 'test']
Samples in validation: 13368

Text length: 4290 characters
Reference: Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation ...

**Load Model and Tokenizer**

In [5]:
print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    local_files_only=False
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    token=HF_TOKEN,
    local_files_only=False
).to("cuda")
model.eval()

print(f"Model loaded successfully!")
print(f"Device: {next(model.parameters()).device}")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"Layers: {len(model.model.layers)}")
print(f"Model size: {sum(p.numel() for p in model.parameters())/1e9:.1f}B parameters")

Loading model: meta-llama/Llama-3.2-1B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 626.54it/s, Materializing param=model.norm.weight]                              


Model loaded successfully!
Device: cuda:0
Dtype: torch.bfloat16
Layers: 16
Model size: 1.2B parameters


If everything goes will you should be able to see following print information:

Loading model: meta-llama/Llama-3.2-1B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 626.54it/s, Materializing param=model.norm.weight]                              
Model loaded successfully!
Device: cuda:0
Dtype: torch.bfloat16
Layers: 16
Model size: 1.2B parameters

**KV Cache Patching (H2O / Random / Reset)**

These functions mirror h20.py exactly. Call `reset_kv_mode()` before switching to a new config to cleanly remove all hooks and restore original attention forwards.

In [ ]:
import types
import torch.nn.functional as F
from transformers import DynamicCache

# KV Cache configurations all possible combinations of H2O vs random eviction, and different budget splits.
KV_CONFIGS = {
    "full":      {"h2o": False, "random": False, "budget_ratio": 1.0, "recent_ratio": 1.0},
    "random_b1": {"h2o": False, "random": True,  "budget_ratio": 0.1, "recent_ratio": 0.1},
    "random_b2": {"h2o": False, "random": True,  "budget_ratio": 0.2, "recent_ratio": 0.1},
    "random_b4": {"h2o": False, "random": True,  "budget_ratio": 0.4, "recent_ratio": 0.1},
    "random_b6": {"h2o": False, "random": True,  "budget_ratio": 0.6, "recent_ratio": 0.1},
    "random_b8": {"h2o": False, "random": True,  "budget_ratio": 0.8, "recent_ratio": 0.1},
    "h2o_b1_r1": {"h2o": True,  "random": False, "budget_ratio": 0.1, "recent_ratio": 0.1},
    "h2o_b2_r1": {"h2o": True,  "random": False, "budget_ratio": 0.2, "recent_ratio": 0.1},
    "h2o_b2_r2": {"h2o": True,  "random": False, "budget_ratio": 0.2, "recent_ratio": 0.2},
    "h2o_b3_r1": {"h2o": True,  "random": False, "budget_ratio": 0.3, "recent_ratio": 0.1},
    "h2o_b3_r2": {"h2o": True,  "random": False, "budget_ratio": 0.3, "recent_ratio": 0.2},
    "h2o_b3_r3": {"h2o": True,  "random": False, "budget_ratio": 0.3, "recent_ratio": 0.3},
    "h2o_b4_r1": {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.1},
    "h2o_b4_r2": {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.2},
    "h2o_b4_r3": {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.3},
    "h2o_b4_r4": {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.4},
    "h2o_b5_r1": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.1},
    "h2o_b5_r2": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.2},
    "h2o_b5_r3": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.3},
    "h2o_b5_r4": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.4},
    "h2o_b5_r5": {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.5},
    "h2o_b6_r1": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.1},
    "h2o_b6_r2": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.2},
    "h2o_b6_r3": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.3},
    "h2o_b6_r4": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.4},
    "h2o_b6_r5": {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.5},
    "h2o_b7_r1": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.1},
    "h2o_b7_r2": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.2},
    "h2o_b7_r3": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.3},
    "h2o_b7_r4": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.4},
    "h2o_b7_r5": {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.5},
    "h2o_b8_r1": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.1},
    "h2o_b8_r2": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.2},
    "h2o_b8_r3": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.3},
    "h2o_b8_r4": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.4},
    "h2o_b8_r5": {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.5},
    "local_b1":  {"h2o": True,  "random": False, "budget_ratio": 0.1, "recent_ratio": 0.1},
    "local_b2":  {"h2o": True,  "random": False, "budget_ratio": 0.2, "recent_ratio": 0.2},
    "local_b3":  {"h2o": True,  "random": False, "budget_ratio": 0.3, "recent_ratio": 0.3},
    "local_b4":  {"h2o": True,  "random": False, "budget_ratio": 0.4, "recent_ratio": 0.4},
    "local_b5":  {"h2o": True,  "random": False, "budget_ratio": 0.5, "recent_ratio": 0.5},
    "local_b6":  {"h2o": True,  "random": False, "budget_ratio": 0.6, "recent_ratio": 0.6},
    "local_b7":  {"h2o": True,  "random": False, "budget_ratio": 0.7, "recent_ratio": 0.7},
    "local_b8":  {"h2o": True,  "random": False, "budget_ratio": 0.8, "recent_ratio": 0.8},
}

NUM_Q_HEADS  = 32
NUM_KV_HEADS = 8
GQA_GROUP    = NUM_Q_HEADS // NUM_KV_HEADS

# --- State tracking for active hooks, just incase we want to try different configurations, we need to save previous hook ---
_active_state = {"model_hook": None, "orig_forwards": None, "reset_scores_fn": None}

def apply_h2o(model, budget_ratio=0.2, recent_ratio=0.1, cache_size=CACHE_SIZE, strategy="per_head"):
    num_layers    = len(model.model.layers)
    acc_scores    = [None] * num_layers
    step_counter  = [0]
    TOTAL_BUDGET  = max(16, int(cache_size * budget_ratio))
    RECENT_BUDGET = max(8,  int(cache_size * recent_ratio))
    HH_BUDGET     = TOTAL_BUDGET - RECENT_BUDGET
    print(f"H2O Strategy={strategy} | Total Budget: {TOTAL_BUDGET} | HH: {HH_BUDGET} | Recent: {RECENT_BUDGET}")

    def reset_scores():
        for li in range(num_layers):
            acc_scores[li] = None
        step_counter[0] = 0

    def make_patched_forward(orig, li):
        def patched_forward(self, hidden_states, **kwargs):
            kwargs["output_attentions"] = True
            out = orig(hidden_states, **kwargs)
            attn_w = out[1] if isinstance(out, tuple) else getattr(out, "attentions", None)
            if attn_w is not None:
                raw_score = attn_w.float().mean(dim=2)[0]
                score = raw_score.view(NUM_KV_HEADS, GQA_GROUP, -1).mean(dim=1)
                score_gpu = score.detach()
                if acc_scores[li] is None:
                    acc_scores[li] = score_gpu
                else:
                    cur_len = acc_scores[li].shape[1]
                    new_len = score_gpu.shape[1]
                    if new_len > cur_len:
                        pad = torch.zeros(NUM_KV_HEADS, new_len - cur_len, device=score_gpu.device)
                        acc_scores[li] = torch.cat([acc_scores[li], pad], dim=1)
                    acc_scores[li][:, :new_len] += score_gpu
            if isinstance(out, tuple):
                return (out[0], None) + out[2:]
            return out
        return patched_forward

    orig_forwards = {}
    for layer_idx, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        orig_forwards[layer_idx] = attn.forward
        attn.forward = types.MethodType(make_patched_forward(attn.forward, layer_idx), attn)

    def eviction_hook(module, input, output):
        past_kv = getattr(output, "past_key_values", None)
        if past_kv is None or not hasattr(past_kv, 'layers'):
            return output
        for li in range(len(past_kv.layers)):
            if acc_scores[li] is None:
                continue
            k = past_kv.layers[li].keys
            v = past_kv.layers[li].values
            bsz, n_heads, seq_len, head_dim = k.shape
            if seq_len <= TOTAL_BUDGET:
                continue
            device       = k.device
            recent_start = seq_len - RECENT_BUDGET
            scores       = acc_scores[li]
            old_scores   = scores[:, :recent_start]
            if strategy == "layer_shared":
                shared_scores = old_scores.mean(dim=0)
                shared_hh_idx = shared_scores.topk(HH_BUDGET).indices
                hh_idx = shared_hh_idx.unsqueeze(0).expand(n_heads, -1)
            else:
                hh_idx = old_scores.topk(HH_BUDGET, dim=1).indices
            recent_idx = torch.arange(recent_start, seq_len, device=device).unsqueeze(0).expand(n_heads, -1)
            keep_idx, _ = torch.cat([hh_idx, recent_idx], dim=1).sort(dim=1)
            gather_idx = keep_idx.unsqueeze(0).unsqueeze(-1).expand(bsz, -1, -1, head_dim)
            past_kv.layers[li].keys   = torch.gather(k, 2, gather_idx)
            past_kv.layers[li].values = torch.gather(v, 2, gather_idx)
            acc_scores[li] = torch.gather(scores, 1, keep_idx)
        return output

    model_hook = model.model.register_forward_hook(eviction_hook)
    print(f"H2O applied ({strategy}): {num_layers} layers patched.")
    return model_hook, reset_scores, orig_forwards


def apply_random(model, budget_ratio=0.2, cache_size=CACHE_SIZE):
    num_layers   = len(model.model.layers)
    TOTAL_BUDGET = max(16, int(cache_size * budget_ratio))
    print(f"Random Eviction | Total Budget: {TOTAL_BUDGET} tokens")

    def eviction_hook(module, input, output):
        past_kv = getattr(output, "past_key_values", None)
        if past_kv is None or not hasattr(past_kv, 'layers'):
            return output
        for li in range(len(past_kv.layers)):
            k = past_kv.layers[li].keys
            v = past_kv.layers[li].values
            bsz, n_heads, seq_len, head_dim = k.shape
            if seq_len <= TOTAL_BUDGET:
                continue
            device   = k.device
            perm     = torch.randperm(seq_len, device=device)[:TOTAL_BUDGET]
            keep_idx = perm.sort().values.unsqueeze(0).expand(n_heads, -1)
            gather_idx = keep_idx.unsqueeze(0).unsqueeze(-1).expand(bsz, -1, -1, head_dim)
            past_kv.layers[li].keys   = torch.gather(k, 2, gather_idx)
            past_kv.layers[li].values = torch.gather(v, 2, gather_idx)
        return output

    model_hook = model.model.register_forward_hook(eviction_hook)
    print(f"Random eviction applied: {num_layers} layers patched.")
    return model_hook, lambda: None


def restore_attn_forwards(model, orig_forwards):
    """Undo the self_attn.forward patches applied by apply_h2o."""
    for layer_idx, layer in enumerate(model.model.layers):
        if layer_idx in orig_forwards:
            layer.self_attn.forward = orig_forwards[layer_idx]
    print("Attention forwards restored.")


def reset_kv_mode(model):
    """Remove all active hooks and restore attention forwards. Call before switching config."""
    if _active_state["model_hook"] is not None:
        _active_state["model_hook"].remove()
        _active_state["model_hook"] = None
        print("✓ Eviction hook removed.")
    if _active_state["orig_forwards"] is not None:
        restore_attn_forwards(model, _active_state["orig_forwards"])
        _active_state["orig_forwards"] = None
    if _active_state["reset_scores_fn"] is not None:
        _active_state["reset_scores_fn"]()
        _active_state["reset_scores_fn"] = None
    print("✓ Model reset to clean state.")


def apply_kv_mode(model, kv_mode, strategy="per_head", cache_size=CACHE_SIZE):
    """Apply a KV cache mode to the model. Always call reset_kv_mode() first."""
    if kv_mode not in KV_CONFIGS:
        raise ValueError(f"Unknown mode '{kv_mode}'. Available: {list(KV_CONFIGS.keys())}")
    kv_cfg = KV_CONFIGS[kv_mode]

    if kv_cfg["h2o"]:
        hook, reset_fn, orig_fwd = apply_h2o(
            model,
            budget_ratio=kv_cfg["budget_ratio"],
            recent_ratio=kv_cfg["recent_ratio"],
            cache_size=cache_size,
            strategy=strategy,
        )
        _active_state["model_hook"]      = hook
        _active_state["reset_scores_fn"] = reset_fn
        _active_state["orig_forwards"]   = orig_fwd
    elif kv_cfg["random"]:
        hook, reset_fn = apply_random(model, budget_ratio=kv_cfg["budget_ratio"], cache_size=cache_size)
        _active_state["model_hook"]      = hook
        _active_state["reset_scores_fn"] = reset_fn
        _active_state["orig_forwards"]   = None
    else:
        print("KV Mode: full cache — no eviction applied.")
        _active_state["model_hook"]      = None
        _active_state["reset_scores_fn"] = None
        _active_state["orig_forwards"]   = None

    TOTAL_BUDGET  = max(16, int(cache_size * kv_cfg["budget_ratio"]))
    RECENT_BUDGET = max(8,  int(cache_size * kv_cfg["recent_ratio"]))
    print(f"\nActive mode : {kv_mode}  ({strategy})")
    print(f"Cache Size  : {cache_size}")
    print(f"Total Budget: {TOTAL_BUDGET} ({kv_cfg['budget_ratio']*100:.0f}%)")
    print(f"Recent      : {RECENT_BUDGET} ({kv_cfg['recent_ratio']*100:.0f}%)")
    print(f"HH Budget   : {TOTAL_BUDGET - RECENT_BUDGET} ({(kv_cfg['budget_ratio']-kv_cfg['recent_ratio'])*100:.0f}%)")
    return kv_cfg

print("✓ KV cache patch functions defined.")
print("  Usage:")
print("    reset_kv_mode(model)                     # always call first when changing config")
print("    kv_cfg = apply_kv_mode(model, KV_MODE, H2O_STRATEGY, CACHE_SIZE)")

✓ KV cache patch functions defined.
  Usage:
    reset_kv_mode(model)                     # always call first when changing config
    kv_cfg = apply_kv_mode(model, KV_MODE, H2O_STRATEGY, CACHE_SIZE)


If everything goes will you should be able to see following print information:

✓ KV cache patch functions defined.
  Usage:
    reset_kv_mode(model)                     # always call first when changing config
    kv_cfg = apply_kv_mode(model, KV_MODE, H2O_STRATEGY, CACHE_SIZE)

## Metrics & Evaluation

In [7]:
def is_chinese(text):
    for char in text:
        if '\u4e00' <= char <= '\u9fff':
            return True
    return False

def compute_char_level_rouge(pred, ref):
    pred_chars = list(pred.strip())
    ref_chars = list(ref.strip())
    m, n = len(pred_chars), len(ref_chars)
    if not m or not n:
        return 0.0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_chars[i - 1] == ref_chars[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs_len = dp[m][n]
    recall = lcs_len / n if n > 0 else 0.0
    precision = lcs_len / m if m > 0 else 0.0
    if recall + precision == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

try:
    from rouge_score import rouge_scorer as _rs
    _scorer = _rs.RougeScorer(['rougeL'], use_stemmer=False)
    def compute_rouge_l(pred, ref):
        if not pred.strip() or not ref.strip():
            return 0.0
        if is_chinese(ref):
            return compute_char_level_rouge(pred, ref)
        return _scorer.score(ref, pred)['rougeL'].fmeasure
    print("✓ ROUGE-L: rouge_score library (with character-level support for Chinese)")
except ImportError:
    def compute_rouge_l(pred, ref):
        if not pred.strip() or not ref.strip():
            return 0.0
        if is_chinese(ref):
            return compute_char_level_rouge(pred, ref)
        return 0.0
    print("rouge_score not installed, using fallback")

✓ ROUGE-L: rouge_score library (with character-level support for Chinese)


**Run Generation**

Applies the configured KV cache mode, generates, then resets hooks automatically.

In [ ]:

import contextlib, io

# template prompt
prompt = cfg["prompt_template"].format(text=text)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
input_token_count = inputs["input_ids"].shape[1]
print(f"Prompt + Text tokens: {input_token_count}")
print(f"Running {len(CONFIGS)} config(s)...\n")

all_results = []

for run_idx, run_cfg in enumerate(CONFIGS):
    kv_mode  = run_cfg["kv_mode"]
    strategy = run_cfg["h2o_strategy"]

    with contextlib.redirect_stdout(io.StringIO()):
        reset_kv_mode(model)
        apply_kv_mode(model, kv_mode, strategy, CACHE_SIZE)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t_start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    latency  = time.time() - t_start
    mem_peak = torch.cuda.max_memory_allocated() / 1024**2

    with contextlib.redirect_stdout(io.StringIO()):
        reset_kv_mode(model)

    prediction    = tokenizer.decode(outputs[0][input_token_count:], skip_special_tokens=True).strip()
    output_tokens = outputs.shape[1] - input_token_count
    rouge_l       = compute_rouge_l(prediction, reference_str)

    print(f"[{run_idx+1}/{len(CONFIGS)}] {kv_mode} ({strategy})  "
          f"ROUGE-L={rouge_l:.4f}  {latency:.1f}s  {mem_peak:.0f}MB")

    all_results.append({
        "kv_mode":       kv_mode,
        "h2o_strategy":  strategy,
        "rouge_l":       rouge_l,
        "latency_s":     latency,
        "mem_peak_mb":   mem_peak,
        "input_tokens":  input_token_count,
        "output_tokens": output_tokens,
        "prediction":    prediction,
    })

print(f"\n✓ Done — {len(all_results)} configs completed.")


Prompt + Text tokens: 948
Running 5 config(s)...

[1/5] full (per_head)  ROUGE-L=0.1714  2.5s  2719MB
[2/5] random_b6 (per_head)  ROUGE-L=0.0588  1.5s  2719MB
[3/5] h2o_b5_r1 (per_head)  ROUGE-L=0.1404  1.4s  2719MB
[4/5] h2o_b7_r2 (per_layer)  ROUGE-L=0.1538  1.1s  2719MB
[5/5] h2o_b7_r2 (per_head)  ROUGE-L=0.1538  1.1s  2719MB

✓ Done — 5 configs completed.


If everything goes will you should be able to see following print information:

Prompt + Text tokens: 948
Running 5 config(s)...

[1/5] full (per_head)  ROUGE-L=0.1714  2.5s  2719MB
[2/5] random_b6 (per_head)  ROUGE-L=0.0588  1.5s  2719MB
[3/5] h2o_b5_r1 (per_head)  ROUGE-L=0.1404  1.4s  2719MB
[4/5] h2o_b7_r2 (per_layer)  ROUGE-L=0.1538  1.1s  2719MB
[5/5] h2o_b7_r2 (per_head)  ROUGE-L=0.1538  1.1s  2719MB

✓ Done — 5 configs completed.

**Summary & Results**

In [ ]:
#Reference
print("=" * 70)
print(f"DATASET:   {DATASET}  |  sample={SAMPLE_IDX}  |  split={SPLIT}")
print("=" * 70)
print("REFERENCE (ground truth highlights):")
print(reference_str)
print()

#Comparison table
col_w = 22
header = (
    f"{'KV Mode':<{col_w}} {'Strategy':<14} {'ROUGE-L':>8} "
    f"{'Latency':>9} {'Mem(MB)':>9} {'In Tok':>7} {'Out Tok':>8}"
)
print(header)
print("-" * len(header))

for r in all_results:
    print(
        f"{r['kv_mode']:<{col_w}} {r['h2o_strategy']:<14} {r['rouge_l']:>8.4f} "
        f"{r['latency_s']:>8.2f}s {r['mem_peak_mb']:>9.0f} "
        f"{r['input_tokens']:>7} {r['output_tokens']:>8}"
    )

#Per-config predictions
print()
for r in all_results:
    print(f"{'─'*70}")
    print(f"[{r['kv_mode']} / {r['h2o_strategy']}]  ROUGE-L={r['rouge_l']:.4f}")
    print(r["prediction"])
print(f"{'─'*70}")

DATASET:   cnn_dailymail  |  sample=0  |  split=validation
REFERENCE (ground truth highlights):
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

KV Mode                Strategy        ROUGE-L   Latency   Mem(MB)  In Tok  Out Tok
-----------------------------------------------------------------------------------
full                   per_head         0.1714     2.46s      2719     948      100
random_b6              per_head         0.0588     1.54s      2719     948      100
h2o_b5_r1              per_head         0.1404     1.39s      2719     948      100
h2o_b7_r2              per_layer        0.1538     1.10s      2719     948       80
h2o_b7_r2              per_head         0.1538     1.10s      2719     948       80

──────────────────────────────────────────────────────────────────────
[full / per_head]  ROUGE-L=0.1714
A woman named Zully Broussard selflessly donated one of her kidney

If everything goes will you should be able to see following print information:

======================================================================
DATASET:   cnn_dailymail  |  sample=0  |  split=validation
======================================================================
REFERENCE (ground truth highlights):
Zully Broussard decided to give a kidney to a stranger .
A new computer program helped her donation spur transplants for six kidney patients .

KV Mode                Strategy        ROUGE-L   Latency   Mem(MB)  In Tok  Out Tok
-----------------------------------------------------------------------------------
full                   per_head         0.1714     2.46s      2719     948      100
random_b6              per_head         0.0588     1.54s      2719     948      100
h2o_b5_r1              per_head         0.1404     1.39s      2719     948      100
h2o_b7_r2              per_layer        0.1538     1.10s      2719     948       80
h2o_b7_r2              per_head         0.1538     1.10s      2719     948       80

──────────────────────────────────────────────────────────────────────
[full / per_head]  ROUGE-L=0.1714
A woman named Zully Broussard selflessly donated one of her kidneys to a stranger, which sparked a chain reaction of six transplants, with her kidney being matched with a recipient who was paired with a donor. This rare and complex process, known as paired donation, uses genetic profiles to match compatible donors and recipients, and has been made possible by the use of computer programs like MatchGrid. Broussard's generosity has opened up possibilities for pairing compatible donors and recipients, and has
──────────────────────────────────────────────────────────────────────
[random_b6 / per_head]  ROUGE-L=0.0588
A Differenter. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That's a different. That
──────────────────────────────────────────────────────────────────────
[h2o_b5_r1 / per_head]  ROUGE-L=0.1404
A selfless act of generosity led to a chain reaction of six kidney transplants, with the recipient of one kidney being paired with a donor, and the process taking five surgeons to perform the surgeries. The chain of donations was facilitated by a computer program called Match, which uses genetic data to find compatible matches. The program has been used to match donors and recipients in a rare and complex process called paired donation. The article highlights the power of altruism and the potential of technology to make a significant impact
──────────────────────────────────────────────────────────────────────
...
──────────────────────────────────────────────────────────────────────
[h2o_b7_r2 / per_head]  ROUGE-L=0.1538
A selfless act of generosity led to a chain reaction of six kidney transplants, with the recipient of one kidney being paired with a donor, and the process taking over 40 days to complete. The chain of surgeries was facilitated by a computer program called Match, which uses genetic data to find compatible donor-recipient pairs. The program has the potential to revolutionize the field of organ transplantation.
──────────────────────────────────────────────────────────────────────
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

## Try Different Configurations

To compare different configurations, go back to the **Configuration** cell and edit the `CONFIGS` list, then re-run from the **Run Generation** cell down.

```python
# Example: compare full vs different H2O budgets
CONFIGS = [
    {"kv_mode": "full",      "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b3_r1", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b5_r1", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b7_r2", "h2o_strategy": "per_head"},
]

# Example: compare per_head vs layer_shared
CONFIGS = [
    {"kv_mode": "h2o_b5_r1", "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b5_r1", "h2o_strategy": "layer_shared"},
]

# Example: compare random vs H2O at same budget
CONFIGS = [
    {"kv_mode": "random_b4",  "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b4_r1",  "h2o_strategy": "per_head"},
    {"kv_mode": "h2o_b4_r2",  "h2o_strategy": "per_head"},
]
```

You can also change `DATASET`, `SAMPLE_IDX`, `CACHE_SIZE`, or `MAX_SEQ_LEN` for further experiments. But please pay attention for longbench dataset there is no validation option you can only select test dataset.

## **Conclusions**

Summarize what you could and could not conclude based on your experiments.
In this section, you can add **text**.



## **References**

### References

1.  Zhang, Z., Sheng, Y., Zhou, T., Chen, T., Zheng, L., Cai, R., ... & Chen, B.
(2023). **H2O: Heavy-Hitter Oracle for Efficient Generative Inference of Large Language Models**. *arXiv preprint arXiv:2306.14048*. [https://arxiv.org/abs/2306.14048](https://arxiv.org/abs/2306.14048)

2.  Yilong Chen, Guoxia Wang, Junyuan Shang ... (2023). **NACL: A General and Effective KV Cache Eviction Framework for LLMs at Inference Time**. *arXiv preprint arXiv:2408.03675*. [https://arxiv.org/abs/2408.03675](https://arxiv.org/abs/2408.03675)

3. Li, H., Li, Y., Tian, A., Tang, T., Xu, Z., Chen, X., ... & Chen, L. (2025). **A Survey on Large Language Model Acceleration based on KV Cache Management**. *arXiv preprint arXiv:2412.19442*. [https://arxiv.org/abs/2412.19442](https://arxiv.org/abs/2412.19442)


4. Meta AI. (2024). **Llama 3.2-1B**. Hugging Face. [https://huggingface.co/meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B)

5. A., Liu, P. J., & Manning, C. D. (2017). **CNN/Daily Mail Dataset**. Hugging Face. [https://huggingface.co/datasets/abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail)
   
6. Bai, Yushi et all. (2023). **LongBench Dataset**. Hugging Face. [https://huggingface.co/datasets/yanbingzheng/LongBench](https://huggingface.co/datasets/yanbingzheng/LongBench)